# Extract transect and make radial topo
## Transect into Westport WA

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools, kmltools, dtopotools
from clawpack.visclaw import gridtools, animation_tools
from clawpack.geoclaw.util import haversine, gctransect
from clawpack.geoclaw.data import Rearth
from clawpack.clawutil.util import fullpath_import
from scipy.interpolate import interp1d
from scipy.interpolate import RegularGridInterpolator, griddata
import os,sys
NGT = fullpath_import('/Users/rjl/git/AGTwork/geoclaw1d/src_python/nonuniform_grid_tools2.py')

In [ ]:
deg2m = Rearth*pi/180  # approx 111e3

In [ ]:
extent = [-140, -123.85, 46., 48.]
coarsen = 1
arcsec = coarsen * 30
print('Will download etopo22_30s data at %i arcsecond resolution' % arcsec)

In [ ]:
url_thredds = 'https://www.ngdc.noaa.gov/thredds/dodsC/global/ETOPO2022/30s/30s_bed_elev_netcdf/ETOPO_2022_v1_30s_N90W180_bed.nc'
etopo = topotools.read_netcdf(url_thredds, extent=extent,
                             coarsen=coarsen, verbose=True)

In [ ]:
figure(figsize=(12,6))
ax = axes()
etopo.plot(axes=ax, limits=(-3000,1000),
          cb_kwargs={'extend':'both','shrink':0.7})
title('etopo 2022 30" topo');

In [ ]:
y0 = 46.835

In [ ]:
j0 = where(etopo.y <= y0)[0].max()
y0 = etopo.y[j0]
print(f'Extracting E-W transect at y = {y0:.5f}')

fig,axs = subplots(2,1,figsize=(10,11))

xtrans = etopo.X[j0,:]  # x along transect
ytrans = etopo.Y[j0,:]  # y along transect (constant)
ztrans = etopo.Z[j0,:]  # topo along transect
ax = axs[0]
ax.plot(etopo.x, ztrans, 'g')
ax.grid(True)
#ax.set_ylim(-50,50)
ax.ticklabel_format(useOffset=False)

ax.set_title(f'30" topography on transect at y = {y0:.5f}');


ax = axs[1]
ax.plot(etopo.x, ztrans)

# zoom in on lower plot:

ax = axs[1]
ax.plot(etopo.x, ztrans, 'g')
ax.set_ylim(-10,15)
#ax.set_xlim(-124.4, -123.9)
ax.set_xlim(-124.15, -124.01)
ax.set_xlabel('longitude')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Zoom near shore');


## Nearshore topo

In [ ]:
url_thredds = 'https://www.ngdc.noaa.gov/thredds/dodsC/tiles/nthmp/tiled_19as/' \
    + 'sowa_mhw_19_n47x00_w124x25_2025v1.nc'
extent = [-124.2,-124, 46.82, 46.87]
#coarsen = 3 # subsample from 1/9" to 1/3"
coarsen = 1  # full 1/9" resolution
topo19 = topotools.read_netcdf(url_thredds, extent=extent,
                             coarsen=coarsen, verbose=True)

In [ ]:
figure(figsize=(12,6))
ax = axes()
topo19.plot(axes=ax, limits=(-50,50),
          cb_kwargs={'extend':'both','shrink':0.7})
title('CUDEM 1/9" topo');

In [ ]:
j0 = where(topo19.y <= y0)[0].max()
y0 = topo19.y[j0]
print(f'Extracting E-W transect at y = {y0:.5f}')

fig,axs = subplots(2,1,figsize=(10,11))

xtrans = topo19.X[j0,:]  # x along transect
ytrans = topo19.Y[j0,:]  # y along transect (constant)
ztrans = topo19.Z[j0,:]  # topo along transect
ax = axs[0]
ax.plot(topo19.x, ztrans, 'g')
ax.grid(True)
ax.set_ylim(-40,20)
ax.ticklabel_format(useOffset=False)

ax.set_title(f'1/3" topography on transect at y = {y0:.5f}');

# zoom in on lower plot:

ax = axs[1]
ax.plot(topo19.x, ztrans, 'g')
ax.set_ylim(-10,15)
ax.set_xlim(-124.12, -124.07)
ax.set_xlabel('longitude')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Zoom near shore');

In [ ]:
topos = [etopo, topo19]

In [ ]:
x1trans, x2trans = -128, -124.07
y1trans, y2trans = y0, y0
mxtopo = 200000  # start with finer grid to interpolate from topofiles

xtrans,ytrans = gctransect(x1trans,y1trans, 
                                x2trans,y2trans, mxtopo, 'W')
print(f'Found great circle transect with {len(xtrans)} points')

In [ ]:
ztrans = nan*ones(xtrans.shape)
toposource = -ones(xtrans.shape)

for k,topo in enumerate(topos):
    #topo_fcn =  RegularGridInterpolator((topo.x,topo.y), topo.Z.T,       
    #                     method='linear',bounds_error=False,fill_value=nan)
    #xytrans = vstack((xtrans,ytrans)).T
    topo_fcn = topo.make_function()
    topotrans = topo_fcn(xtrans,ytrans)
    ztrans = where(isnan(topotrans), ztrans, topotrans)
    toposource = where(isnan(topotrans), toposource, k)
    nk = (toposource == k).sum()
    print(f'Extracted {nk} points from topofile {k}')

In [ ]:
if isnan(ztrans).sum() > 0:
    print('*** ztrans has %i nan values' % isnan(ztrans).sum()) 

In [ ]:
figure(figsize=(12,6))
ax = axes()
etopo.plot(axes=ax, limits=(-3000,1000),
           cb_kwargs={'extend':'both','shrink':0.7})
plot(xtrans, ytrans, 'yellow')
xlim(-130, -123.8)
title('Great circle transect');

In [ ]:
# convert to meters along transect:
rtrans = haversine(xtrans[0],ytrans[0],xtrans,ytrans)
print(f'Length of transect = {(rtrans[-1]-rtrans[0])/1e3:.3f} km')


# shift away from origin:
#rtrans += 1000e3

topo_fcn_1d = interp1d(rtrans, ztrans, kind='linear', fill_value='extrapolate')

if 0:
    h_min = 20  # switch to uniform fine grid at this depth 
    dx_min = 1. # grid resolution on and near shore
    #dx_max = 500. # maximum in deep ocean, roughly 15"
    rp,zp = NGT.make_celledges_cfl(r[0], r[-1], topo_fcn_1d,
                                   dx_min=dx_min, h_min=h_min, fname='celledges.data',
                                   plot_topo=False)

### make celledges files at various resolutions

For 1m onshore resolution the resolution is 50 m in the deep ocean.
For coarser resolutions all cell sizes are scaled up.

In [ ]:
xdx_vals_1m = array([[0, 50.], [150e3, 50.], [280e3, 10.], [290e3, 1.], [rtrans[-1], 1.]])
for onshore_res in [10, 5, 2, 1]:
    xdx_vals = xdx_vals_1m.copy()
    xdx_vals[:,1] *= onshore_res
    dx_fcn = interp1d(xdx_vals[:,0], xdx_vals[:,1], kind='linear')
    x2 = rtrans[-1]
    x = zeros(50000)
    for j in range(len(x)):
        x[j+1] = x[j] + dx_fcn(x[j])
        if x[j+1] > x2:
            break
    x = x[:(j+1)]
    onshore_res = xdx_vals[-1,1]
    print(f'Selected {len(x)} points on transect with onshore resolution {onshore_res} m')
    
    redge = x
    zedge = topo_fcn_1d(x)

    rlat = 20 + redge / deg2m  # convert to latitudes relative to pole at x=0
    rlat_z = vstack((rlat,zedge)).T
    fname = f'celledges_{int(onshore_res):d}m.txt'
    savetxt(fname, rlat_z, fmt='%.10f', header=f'{len(rlat)}  # number of cell edges', comments='')
    print('Created ',fname)

In [ ]:
redge[-1] - redge[-2]

In [ ]:
fig,axs = subplots(2,1,figsize=(10,11))

rkm = redge/1e3

ax = axs[0]
ax.plot(rkm, zedge, 'g')
ax.grid(True)
ax.set_title(f'GeoClaw 1D topography on great circle transect with variable spacing');

ax = axs[1]
ax.plot(rkm, zedge, 'g')
ax.set_ylim(-10,20)
ax.set_xlim(rkm[-1]-5, rkm[-1])
ax.set_xlabel('radial distance r (km)')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title(f'Zoom near shore for onshore resolution {onshore_res} m');

## DTopo file

In [ ]:
dtopofile = '/Users/rjl/B/dtopo/dtopofiles/CSZ_L1-extended-pmel.tt3'
dtopo2d = dtopotools.DTopography(dtopofile, 3)

In [ ]:
dtopo2d_fcn = dtopo2d.make_function()

dtopo_trans = dtopo2d_fcn(xtrans, ytrans, 10)
dtopo_fcn_1d = interp1d(rtrans, dtopo_trans, kind='linear', fill_value='extrapolate')

rdtopo = hstack((0., arange(150e3, 300e3, 500)))
dz_L1 = dtopo_fcn_1d(rdtopo)

In [ ]:
plot(rdtopo/1e3, dz_L1)

In [ ]:
fig,axs = subplots(2,1,figsize=(10,11))

rkm = redge/1e3

ax = axs[0]
ax.plot(rdtopo/1e3, dz_L1)
ax.set_xlim(150,300)
ax.set_ylim(-10,20)
ax.set_ylabel('dz (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Deformation');

ax = axs[1]
ax.plot(rkm, zedge, 'g')
ax.grid(True)

#ax.set_xlim(927, 931)
ax.set_xlim(150,300)
ax.set_xlabel('radial distance r (km)')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Topography')


In [ ]:
if 1:
    t_rdtopo_dz = vstack((ones(len(rdtopo)), rdtopo, dz)).T
    fname = 'L1_dtopo.dtt1'
    savetxt(fname, t_rdtopo_dz, fmt='%.10f')

In [ ]:

#xdx_vals_1m = array([[0, 50.], [700e3, 50.], [920e3, 1.], [1100e3, 1.]])
xdx_vals_1m = array([[0, 50.], [150e3, 50.], [280e3, 10.], [290e3, 1.], [rtrans[-1], 1.]])


for onshore_res in [10, 5, 2, 1]:
    xdx_vals = xdx_vals_1m.copy()
    xdx_vals[:,1] *= onshore_res
    dx_fcn = interp1d(xdx_vals[:,0], xdx_vals[:,1], kind='linear')
    x2 = rtrans[-1]
    x = zeros(50000)
    for j in range(len(x)):
        x[j+1] = x[j] + dx_fcn(x[j])
        if x[j+1] > x2:
            break
    x = x[:(j+1)]
    onshore_res = xdx_vals[-1,1]
    print(f'Selected {len(x)} points on transect with onshore resolution {onshore_res} m')
    
    redge = x
    zedge = topo_fcn_1d(x)

    rlat = 20 + redge / deg2m  # convert to latitudes relative to pole at x=0
    rlat_z = vstack((rlat,zedge)).T
    fname = f'celledges_{int(onshore_res):d}m.txt'
    savetxt(fname, rlat_z, fmt='%.10f', header=f'{len(rlat)}  # number of cell edges', comments='')
    print('Created ',fname)

    # dtopo

    dzedge = dtopo_fcn_1d(redge)
    t_rlat_dz = vstack((ones(len(redge)), rlat, dzedge)).T
    fname = f'L1_dtopo_{int(onshore_res):d}m.dtt1'
    savetxt(fname, t_rlat_dz, fmt='%.10f')
    print('Created ',fname)


In [ ]:
plot(rlat, zedge, 'g')
xlim(22.64,22.69)
ylim(-20,20)
grid(True)

In [ ]:
# Slips provided by Yong Wei:
# - Region 2:   35*ac57a+35*ac57b+10*ac58a+33*ac58b+10*ac59a+24*ac59b+30*ac60a+30*ac60b
# - Region 3:   5*ac59a+25*ac59b+30*ac60a+25*ac60b+50*ac61a+40*ac61b


sift_slips = {'acsza57':35, 'acszb57':35 , 'acsza58':10 , 'acszb58':33 ,
              'acsza59':10 , 'acszb59':24 ,
              'acsza60':30 , 'acszb60':30}

fault = dtopotools.SiftFault(sift_slips)
for k in sift_slips.keys():
    s = fault.sift_subfaults[k]  # subfault
    s.longitude = s.longitude - 360.

In [ ]:
x = etopo.x
y = etopo.y
dtopo_SIFT = fault.create_dtopography(x,y, verbose=True)

In [ ]:
# for setting color scale:
dz_max = abs(dtopo_SIFT.dZ).max()
print("maximum abs(dz) over the full rupture time:",     abs(dtopo_SIFT.dZ).max())

fig,ax = subplots(figsize=(12,10))
dtopo_SIFT.plot_dZ_colors(axes=ax, t=1, cmax_dZ=dz_max)
#ax1.plot(shore[:,0], shore[:,1], 'g')
#ax1.plot(x_coast, y_coast, 'g', linewidth=1)
ax.set_xlim(-128,-122)
ax.set_ylim(46,48)
ax.set_aspect(1./cos(47*pi/180.))

In [ ]:
dtopo2d_fcn = dtopo_SIFT.make_function()

dtopo_trans = dtopo2d_fcn(xtrans, ytrans, 10)
dtopo_fcn_1d = interp1d(rtrans, dtopo_trans, kind='linear', fill_value='extrapolate')

rdtopo = hstack((0., arange(150e3, 300e3, 500)))
dz_sift = dtopo_fcn_1d(rdtopo)

In [ ]:
fig,axs = subplots(2,1,figsize=(10,11))

rkm = redge/1e3
xlimits = [120,300]

ax = axs[0]
ax.plot(rdtopo/1e3, dz_sift)
ax.set_xlim(xlimits)
ax.set_ylim(-10,20)
ax.set_ylabel('dz (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Deformation - ASCE SIFT');

ax = axs[1]
ax.plot(rkm, zedge, 'g')
ax.grid(True)

ax.set_xlim(xlimits)
ax.set_xlabel('radial distance r (km)')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Topography')


In [ ]:
x1 = 170e3
width1 = 10e3
amp1 = 5.
x2 = 240e3
width2 = 50e3
amp2 = -0.2*amp1

dtopo_fcn_1d = lambda x: amp1*exp(-((x-x1)/width1)**2) + amp2*exp(-((x-x2)/width2)**2)

dz = dtopo_fcn_1d(redge)

fig,axs = subplots(2,1,figsize=(10,11))

rkm = redge/1e3
xlimits = [120,300]

ax = axs[0]
ax.plot(rkm, dz)
ax.set_xlim(xlimits)
ax.set_ylim(-10,20)
ax.set_ylabel('dz (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Deformation - Gaussians');

ax = axs[1]
ax.plot(rkm, zedge, 'g')
ax.grid(True)

ax.set_xlim(xlimits)
ax.set_xlabel('radial distance r (km)')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Topography')


In [ ]:
for onshore_res in [10, 5, 2, 1]:
    xdx_vals = xdx_vals_1m.copy()
    xdx_vals[:,1] *= onshore_res
    dx_fcn = interp1d(xdx_vals[:,0], xdx_vals[:,1], kind='linear')
    x2 = rtrans[-1]
    x = zeros(50000)
    for j in range(len(x)):
        x[j+1] = x[j] + dx_fcn(x[j])
        if x[j+1] > x2:
            break
    x = x[:(j+1)]
    onshore_res = xdx_vals[-1,1]
    print(f'Selected {len(x)} points on transect with onshore resolution {onshore_res} m')
    
    redge = x
    zedge = topo_fcn_1d(x)

    rlat = 20 + redge / deg2m  # convert to latitudes relative to pole at x=0
    rlat_z = vstack((rlat,zedge)).T
    fname = f'celledges_{int(onshore_res):d}m.txt'
    savetxt(fname, rlat_z, fmt='%.10f', header=f'{len(rlat)}  # number of cell edges', comments='')
    print('Created ',fname)

    # dtopo

    dzedge = dtopo_fcn_1d(redge)
    t_rlat_dz = vstack((ones(len(redge)), rlat, dzedge)).T
    fname = f'gaussian5m_dtopo_{int(onshore_res):d}m.dtt1'
    savetxt(fname, t_rlat_dz, fmt='%.10f')
    print('Created ',fname)

## Make dtopo comparison plots

In [ ]:
x1 = 170e3
width1 = 10e3
amp1 = 10.
x2 = 240e3
width2 = 50e3
amp2 = -0.2*amp1

rdtopo = hstack((0., arange(150e3, 300e3, 500)))
dtopo_fcn_1d = lambda x: amp1*exp(-((x-x1)/width1)**2) + amp2*exp(-((x-x2)/width2)**2)
dz_gaussians = dtopo_fcn_1d(rdtopo)

fig,axs = subplots(2,1,figsize=(10,8),sharex=True)

#rkm = redge/1e3
#xlimits = [120,300]

rdtopo += 20*deg2m  # shift
xlimits = [2300,2550]

ax = axs[0]
ax.plot(rdtopo/1e3, dz_L1, 'b', label=f'L1 on transect')
ax.plot(rdtopo/1e3, dz_sift, 'k', label=f'ASCE SIFT Region 3 on transect')
ax.plot(rdtopo/1e3, dz_gaussians, 'r', label=f'Gaussians with peak amplitude {amp1:.1f} m')

ax.set_xlim(xlimits)
ax.set_ylim(-10,20)
ax.set_ylabel('dz (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Vertical Deformation');
ax.legend(framealpha=1)

ax = axs[1]
ax.plot(rkm + 20*deg2m/1e3, zedge, 'g')
ax.grid(True)

ax.set_xlim(xlimits)
ax.set_xlabel('radial distance r (km)')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Topography');

fname = 'dz_3sources.png'
savefig(fname, bbox_inches='tight')
print('Created ',fname)